<a href="https://colab.research.google.com/github/vikramvundyala/python_AI-ML/blob/main/AIMLBigdata25_Session.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from collections import defaultdict
def map_pahse(records):
  mapped = []
  for text in records:
    words = text.split()
    for word in words:
      mapped.append((word.lower(), 1))
  return mapped

def shuffle(mapped):
  grouped = defaultdict(list)
  for key, value in mapped:
    grouped[key].append(value)
  return grouped

def reduce_phase(grouped):
  return{k:sum(vs) for k ,vs in grouped.items()}

docs = ["hello world","hello spark","spark is fast"]
mapped = map_pahse(docs)
grouped = shuffle(mapped)
reduced = reduce_phase(grouped)
print(reduced)

{'hello': 2, 'world': 1, 'spark': 2, 'is': 1, 'fast': 1}


In [ ]:
!pip install pyspark

In [ ]:
from pyspark import SparkContext
sc = SparkContext.getOrCreate()
text = ["hello world","hello spark","spark is fast"]
rdd = sc.parallelize(text)
word_counts = (rdd
               .flatMap(lambda line:line.split())
               .map(lambda w:(w.lower(),1))
               .reduceByKey(lambda x,y:x+y))
print(word_counts.collect())


[('hello', 2), ('world', 1), ('fast', 1), ('spark', 2), ('is', 1)]


In [ ]:
pages = [

         ('http://site/a',"spark is fast"),
         ('http://site/b',"hadoop is spark"),
         ('http://site/c',"big data with hadoop")

]


rdd = sc.parallelize(pages)

inverted_index = (rdd
                  .flatMap(lambda kv:[(word.lower(),kv[0]) for word in kv[1].split()])
                  .groupByKey()
                  .mapValues(lambda urls:list(set(urls)))
                  )
for word,urls in inverted_index.collect():
    print(word ,"- >" ,urls)

fast - > ['http://site/a']
hadoop - > ['http://site/b', 'http://site/c']
big - > ['http://site/c']
with - > ['http://site/c']
spark - > ['http://site/b', 'http://site/a']
is - > ['http://site/b', 'http://site/a']
data - > ['http://site/c']


In [ ]:
from pyspark.sql import functions as f
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
df = spark.range(0,10_000_000).withColumn("value",(f.rand()*100).cast("int"))
df.cache().count()

df.groupBy("value").count().show()
df.filter(df.value >50).count()

+-----+------+
|value| count|
+-----+------+
|   31| 99751|
|   85| 99749|
|   65|100087|
|   53|100309|
|   78|100505|
|   34| 99693|
|   81| 99905|
|   28| 99793|
|   76| 99922|
|   26| 99886|
|   27| 99636|
|   44|100105|
|   12| 99795|
|   91| 99838|
|   22|100284|
|   93|100079|
|   47|100451|
|    1| 99376|
|   52| 99888|
|   13|100269|
+-----+------+
only showing top 20 rows



4901066

In [ ]:
df.groupBy("value").count().show()
df.filter(df.value >50).count()

+-----+------+
|value| count|
+-----+------+
|   31| 99751|
|   85| 99749|
|   65|100087|
|   53|100309|
|   78|100505|
|   34| 99693|
|   81| 99905|
|   28| 99793|
|   76| 99922|
|   26| 99886|
|   27| 99636|
|   44|100105|
|   12| 99795|
|   91| 99838|
|   22|100284|
|   93|100079|
|   47|100451|
|    1| 99376|
|   52| 99888|
|   13|100269|
+-----+------+
only showing top 20 rows



4901066

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SimpleDemo").getOrCreate()
sc = spark.sparkContext
print(sc.appName)

pyspark-shell


In [ ]:
rdd = sc.parallelize([1,2,3,4,5],numSlices=3)
print("Partition ",rdd.getNumPartitions())
print("Collect ",rdd.collect())

Partition  3
Collect  [1, 2, 3, 4, 5]


In [ ]:
rdd = sc.parallelize(range(10))  # 0,1,2,3,4,5,6,7,8,9
rdd2 = rdd.map(lambda x:x*2)   #0,1,4,9,16,25,36,49,64,81
rdd3 = rdd2.filter(lambda x: x>10)
print(rdd3.count())

4


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode,split
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
lines = (spark.readStream
         .format("socket")
         .option("host","localhost")
         .option("port","9999")
         .load()

)



In [ ]:
words = lines.select(explode(split(lines.value," ")).alias("word"))
wordCounts = words.groupBy("word").count()
query = (wordCounts.writeStream.format("console").outputMode("complete").start())


In [ ]:
!wget https://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/Iris.csv

--2025-11-22 10:43:01--  https://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/Iris.csv
Resolving cdn.iiith.talentsprint.com (cdn.iiith.talentsprint.com)... 172.105.52.210
Connecting to cdn.iiith.talentsprint.com (cdn.iiith.talentsprint.com)|172.105.52.210|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5107 (5.0K) [application/octet-stream]
Saving to: ‘Iris.csv’

Iris.csv            100%[===================>]   4.99K  --.-KB/s    in 0s      

2025-11-22 10:43:02 (72.7 MB/s) - ‘Iris.csv’ saved [5107/5107]



In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("IrisDemo").getOrCreate()

In [ ]:
df = spark.read.csv('/content/Iris.csv',header=True,inferSchema=True)
df.show(5)
df.printSchema()

+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|
+---+-------------+------------+-------------+------------+-----------+
only showing top 5 rows

root
 |-- Id: integer (nullable = true)
 |-- SepalLengthCm: double (nullable = true)
 |-- SepalWidthCm: double (nullable = true)
 |-- PetalLengthCm: double (nullable = true)
 |-- PetalWidthCm: double (nullable = true)
 |-- Species: string (nullable = true)



In [ ]:
df.describe().show()

+-------+------------------+------------------+-------------------+------------------+------------------+--------------+
|summary|                Id|     SepalLengthCm|       SepalWidthCm|     PetalLengthCm|      PetalWidthCm|       Species|
+-------+------------------+------------------+-------------------+------------------+------------------+--------------+
|  count|               150|               150|                150|               150|               150|           150|
|   mean|              75.5| 5.843333333333335| 3.0540000000000007|3.7586666666666693|1.1986666666666672|          NULL|
| stddev|43.445367992456916|0.8280661279778637|0.43359431136217375| 1.764420419952262|0.7631607417008414|          NULL|
|    min|                 1|               4.3|                2.0|               1.0|               0.1|   Iris-setosa|
|    max|               150|               7.9|                4.4|               6.9|               2.5|Iris-virginica|
+-------+------------------+----

In [ ]:
df.count()

150

In [ ]:
df.filter(df.Species == "Iris-setosa").count()

50

In [ ]:
df.filter(df['Species'] =="Iris-setosa" ).show()

+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|
|  6|          5.4|         3.9|          1.7|         0.4|Iris-setosa|
|  7|          4.6|         3.4|          1.4|         0.3|Iris-setosa|
|  8|          5.0|         3.4|          1.5|         0.2|Iris-setosa|
|  9|          4.4|         2.9|          1.4|         0.2|Iris-setosa|
| 10|          4.9|         3.1|          1.5|         0.1|Iris-setosa|
| 11|          5.4|         3.7|          1.5|         0.2|Iris-

In [ ]:
pddf = df.toPandas()
pddf.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [ ]:
df.groupBy('Species').count().show()

+---------------+-----+
|        Species|count|
+---------------+-----+
| Iris-virginica|   50|
|    Iris-setosa|   50|
|Iris-versicolor|   50|
+---------------+-----+



In [ ]:
df.freqItems(['Species']).show()

+--------------------+
|   Species_freqItems|
+--------------------+
|[Iris-virginica, ...|
+--------------------+



In [ ]:
df.tail(1)

[Row(Id=150, SepalLengthCm=5.9, SepalWidthCm=3.0, PetalLengthCm=5.1, PetalWidthCm=1.8, Species='Iris-virginica')]

In [ ]:
df.orderBy(df['Species'].desc()).show()

+---+-------------+------------+-------------+------------+--------------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|       Species|
+---+-------------+------------+-------------+------------+--------------+
|101|          6.3|         3.3|          6.0|         2.5|Iris-virginica|
|121|          6.9|         3.2|          5.7|         2.3|Iris-virginica|
|102|          5.8|         2.7|          5.1|         1.9|Iris-virginica|
|103|          7.1|         3.0|          5.9|         2.1|Iris-virginica|
|104|          6.3|         2.9|          5.6|         1.8|Iris-virginica|
|105|          6.5|         3.0|          5.8|         2.2|Iris-virginica|
|106|          7.6|         3.0|          6.6|         2.1|Iris-virginica|
|107|          4.9|         2.5|          4.5|         1.7|Iris-virginica|
|108|          7.3|         2.9|          6.3|         1.8|Iris-virginica|
|109|          6.7|         2.5|          5.8|         1.8|Iris-virginica|
|110|          7.2|      

In [ ]:
df.createOrReplaceGlobalTempView("iris")

In [ ]:
df_renamed = df.withColumnRenamed('Species','species')\
            .withColumnRenamed('SepalLengthCm','sepal_length_cm')

In [ ]:
df_renamed.show()

+---+---------------+------------+-------------+------------+-----------+
| Id|sepal_length_cm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    species|
+---+---------------+------------+-------------+------------+-----------+
|  1|            5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|            4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  3|            4.7|         3.2|          1.3|         0.2|Iris-setosa|
|  4|            4.6|         3.1|          1.5|         0.2|Iris-setosa|
|  5|            5.0|         3.6|          1.4|         0.2|Iris-setosa|
|  6|            5.4|         3.9|          1.7|         0.4|Iris-setosa|
|  7|            4.6|         3.4|          1.4|         0.3|Iris-setosa|
|  8|            5.0|         3.4|          1.5|         0.2|Iris-setosa|
|  9|            4.4|         2.9|          1.4|         0.2|Iris-setosa|
| 10|            4.9|         3.1|          1.5|         0.1|Iris-setosa|
| 11|            5.4|         3.7|    

In [ ]:
df.show(5)

+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|
+---+-------------+------------+-------------+------------+-----------+
only showing top 5 rows



In [ ]:
from pyspark.ml.feature import VectorAssembler
numericColumns = ['SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm']
assembler = VectorAssembler(inputCols=numericColumns,outputCol="features")
df = assembler.transform(df)
df.show()

+---+-------------+------------+-------------+------------+-----------+-----------------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|         features|
+---+-------------+------------+-------------+------------+-----------+-----------------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|[5.1,3.5,1.4,0.2]|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|[4.9,3.0,1.4,0.2]|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|[4.7,3.2,1.3,0.2]|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|[4.6,3.1,1.5,0.2]|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|[5.0,3.6,1.4,0.2]|
|  6|          5.4|         3.9|          1.7|         0.4|Iris-setosa|[5.4,3.9,1.7,0.4]|
|  7|          4.6|         3.4|          1.4|         0.3|Iris-setosa|[4.6,3.4,1.4,0.3]|
|  8|          5.0|         3.4|          1.5|         0.2|Iris-setosa|[5.0,3.4,1.5,0.2]|
|  9|     

In [ ]:
from pyspark.ml.feature import StringIndexer
label_string = StringIndexer(inputCol="Species", outputCol="label")
df = label_string.fit(df).transform(df)
df.show()


+---+-------------+------------+-------------+------------+-----------+-----------------+-----+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|         features|label|
+---+-------------+------------+-------------+------------+-----------+-----------------+-----+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|[5.1,3.5,1.4,0.2]|  0.0|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|[4.9,3.0,1.4,0.2]|  0.0|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|[4.7,3.2,1.3,0.2]|  0.0|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|[4.6,3.1,1.5,0.2]|  0.0|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|[5.0,3.6,1.4,0.2]|  0.0|
|  6|          5.4|         3.9|          1.7|         0.4|Iris-setosa|[5.4,3.9,1.7,0.4]|  0.0|
|  7|          4.6|         3.4|          1.4|         0.3|Iris-setosa|[4.6,3.4,1.4,0.3]|  0.0|
|  8|          5.0|         3.4|        

In [ ]:
train,test = df.randomSplit([0.7,0.3],seed=2021)

In [ ]:
print(f"train set count {train.count()}")
print(f"test set count {test.count()}")

train set count 109
test set count 41


In [ ]:
from pyspark.ml.classification import LogisticRegression
lr = LogisticRegression(featuresCol="features",labelCol="label")
model = lr.fit(train)

In [ ]:
predictions = model.transform(test)

In [ ]:
predictions.select("label","prediction").show(10)

+-----+----------+
|label|prediction|
+-----+----------+
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
+-----+----------+
only showing top 10 rows



In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator = MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName='accuracy')

In [ ]:
accuracy = evaluator.evaluate(predictions)
print(f"Accuracy:{accuracy}")

Accuracy:1.0


In [ ]:
new_sample =[5.1,2.3,4.5,6.1]
new_data = spark.createDataFrame([new_sample],['SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm'])

In [ ]:
new_data.show()

+-------------+------------+-------------+------------+
|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|
+-------------+------------+-------------+------------+
|          5.1|         2.3|          4.5|         6.1|
+-------------+------------+-------------+------------+

